# LSTM与GRU: 门控循环单元

本notebook介绍改进的RNN架构:
- **梯度问题**: 消失/爆炸的根本原因
- **LSTM**: 长短期记忆网络
- **GRU**: 门控循环单元
- **PyTorch实现**: 简洁高效的实现

LSTM/GRU是处理长序列的标准选择!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

---

## 第一部分: RNN的梯度问题

### 1.1 梯度消失问题

**RNN反向传播**:
$$\frac{\partial L}{\partial \mathbf{W}_{hh}} = \sum_{t=1}^T \frac{\partial L_t}{\partial \mathbf{W}_{hh}}$$

**链式法则展开**:
$$\frac{\partial L_t}{\partial \mathbf{W}_{hh}} = \sum_{k=1}^t \frac{\partial L_t}{\partial \mathbf{h}_t} \prod_{i=k+1}^t \frac{\partial \mathbf{h}_i}{\partial \mathbf{h}_{i-1}} \frac{\partial \mathbf{h}_k}{\partial \mathbf{W}_{hh}}$$

**关键项**:
$$\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \text{diag}(\phi'(\mathbf{h}_{t-1})) \mathbf{W}_{hh}^T$$

**连乘效应**:
$$\prod_{i=k+1}^t \frac{\partial \mathbf{h}_i}{\partial \mathbf{h}_{i-1}} \approx \prod_{i=k+1}^t \mathbf{W}_{hh}^T$$

### 1.2 数值示例

In [ ]:
# 模拟梯度传播
def simulate_gradient_flow(W_value, num_steps):
    """模拟梯度在RNN中的传播"""
    gradient = 1.0
    gradients = [gradient]
    
    for t in range(num_steps):
        gradient *= W_value
        gradients.append(gradient)
    
    return gradients


# 不同权重值的影响
num_steps = 50
W_values = [0.5, 0.9, 1.0, 1.1, 2.0]

plt.figure(figsize=(12, 4))

for W in W_values:
    grads = simulate_gradient_flow(W, num_steps)
    plt.plot(grads, label=f'W={W}')

plt.xlabel('Time Steps', fontsize=12)
plt.ylabel('Gradient Magnitude', fontsize=12)
plt.title('Gradient Flow in RNN', fontsize=14)
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

print("观察:")
print("- W < 1: 梯度消失(指数衰减)")
print("- W = 1: 梯度稳定")
print("- W > 1: 梯度爆炸(指数增长)")

### 1.3 梯度问题的后果

**梯度消失** ($\|\mathbf{W}_{hh}\| < 1$):
- ❌ 长期依赖学不到
- ❌ 早期信息丢失
- ❌ 训练困难

**梯度爆炸** ($\|\mathbf{W}_{hh}\| > 1$):
- ❌ 参数更新剧烈
- ❌ 训练不稳定
- ✅ 可用梯度裁剪缓解

### 1.4 解决方案

**短期方案**:
- 梯度裁剪(对抗爆炸)
- 权重初始化

**长期方案**:
- **LSTM**: 记忆元 + 门控
- **GRU**: 简化的门控
- **残差连接**: 梯度高速公路

---

## 第二部分: LSTM (Long Short-Term Memory)

### 2.1 LSTM的核心思想

**问题**: RNN的隐状态每步都被覆盖

**LSTM方案**: 引入**记忆元** $\mathbf{C}_t$
- 记忆元: 长期信息存储
- 隐状态: 短期信息传递
- 门控: 控制信息流动

### 2.2 LSTM的三个门

**1. 遗忘门** (Forget Gate): 忘记什么?
$$\mathbf{F}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xf} + \mathbf{H}_{t-1} \mathbf{W}_{hf} + \mathbf{b}_f)$$

- $\mathbf{F}_t \approx 0$: 忘记旧信息
- $\mathbf{F}_t \approx 1$: 保留旧信息

**2. 输入门** (Input Gate): 记住什么?
$$\mathbf{I}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xi} + \mathbf{H}_{t-1} \mathbf{W}_{hi} + \mathbf{b}_i)$$

**候选记忆元**:
$$\tilde{\mathbf{C}}_t = \tanh(\mathbf{X}_t \mathbf{W}_{xc} + \mathbf{H}_{t-1} \mathbf{W}_{hc} + \mathbf{b}_c)$$

**3. 输出门** (Output Gate): 输出什么?
$$\mathbf{O}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xo} + \mathbf{H}_{t-1} \mathbf{W}_{ho} + \mathbf{b}_o)$$

### 2.3 LSTM更新规则

**记忆元更新**:
$$\mathbf{C}_t = \mathbf{F}_t \odot \mathbf{C}_{t-1} + \mathbf{I}_t \odot \tilde{\mathbf{C}}_t$$

- $\mathbf{F}_t \odot \mathbf{C}_{t-1}$: 保留多少旧记忆
- $\mathbf{I}_t \odot \tilde{\mathbf{C}}_t$: 添加多少新信息

**隐状态输出**:
$$\mathbf{H}_t = \mathbf{O}_t \odot \tanh(\mathbf{C}_t)$$

### 2.4 LSTM为什么有效?

**梯度流动**:
$$\frac{\partial \mathbf{C}_t}{\partial \mathbf{C}_{t-1}} = \mathbf{F}_t$$

- 如果$\mathbf{F}_t \approx 1$, 梯度畅通无阻!
- 相当于**梯度高速公路**
- 可以学习几百步的长期依赖

### 2.5 从零实现LSTM

In [ ]:
def get_lstm_params(vocab_size, num_hiddens, device):
    """初始化LSTM参数"""
    num_inputs = num_outputs = vocab_size
    
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    
    def three():
        """三组参数: 输入门、遗忘门、输出门"""
        return (normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))
    
    # 输入门
    W_xi, W_hi, b_i = three()
    # 遗忘门
    W_xf, W_hf, b_f = three()
    # 输出门
    W_xo, W_ho, b_o = three()
    # 候选记忆元
    W_xc, W_hc, b_c = three()
    
    # 输出层
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    
    # 附加梯度
    params = [W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o,
              W_xc, W_hc, b_c, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params


def init_lstm_state(batch_size, num_hiddens, device):
    """初始化LSTM状态: (隐状态, 记忆元)"""
    return (torch.zeros((batch_size, num_hiddens), device=device),
            torch.zeros((batch_size, num_hiddens), device=device))


def lstm(inputs, state, params):
    """LSTM前向传播"""
    [W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o,
     W_xc, W_hc, b_c, W_hq, b_q] = params
    
    (H, C) = state  # 隐状态和记忆元
    outputs = []
    
    for X in inputs:
        # 输入门
        I = torch.sigmoid(torch.mm(X, W_xi) + torch.mm(H, W_hi) + b_i)
        # 遗忘门
        F = torch.sigmoid(torch.mm(X, W_xf) + torch.mm(H, W_hf) + b_f)
        # 输出门
        O = torch.sigmoid(torch.mm(X, W_xo) + torch.mm(H, W_ho) + b_o)
        # 候选记忆元
        C_tilde = torch.tanh(torch.mm(X, W_xc) + torch.mm(H, W_hc) + b_c)
        
        # 更新记忆元
        C = F * C + I * C_tilde
        # 更新隐状态
        H = O * torch.tanh(C)
        
        # 输出
        Y = torch.mm(H, W_hq) + b_q
        outputs.append(Y)
    
    return torch.cat(outputs, dim=0), (H, C)


# 测试LSTM
vocab_size = 1000
num_hiddens = 256
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

params = get_lstm_params(vocab_size, num_hiddens, device)
print(f"LSTM参数数量: {sum(p.numel() for p in params):,}")
print(f"对比RNN参数: {vocab_size * num_hiddens * 2 + num_hiddens * num_hiddens:,}")
print(f"LSTM约为RNN的4倍参数量")

### 2.6 PyTorch的LSTM

In [ ]:
class LSTMModel(nn.Module):
    """LSTM语言模型"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0.5):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        
        # Embedding层
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # LSTM层
        self.lstm = nn.LSTM(embed_size, num_hiddens, num_layers,
                            dropout=dropout, batch_first=False)
        
        # 输出层
        self.fc = nn.Linear(num_hiddens, vocab_size)
    
    def forward(self, x, state=None):
        # x: (seq_len, batch)
        x = self.embedding(x)  # (seq_len, batch, embed)
        
        if state is None:
            output, state = self.lstm(x)
        else:
            output, state = self.lstm(x, state)
        
        # output: (seq_len, batch, hidden)
        output = self.fc(output)  # (seq_len, batch, vocab)
        return output, state
    
    def begin_state(self, batch_size, device):
        """初始化状态: (h_0, c_0)"""
        return (torch.zeros((self.num_layers, batch_size, self.num_hiddens), device=device),
                torch.zeros((self.num_layers, batch_size, self.num_hiddens), device=device))


# 创建模型
model = LSTMModel(vocab_size=1000, embed_size=128, 
                  num_hiddens=256, num_layers=2, dropout=0.5)
print(model)
print(f"\n总参数量: {sum(p.numel() for p in model.parameters()):,}")

# 测试前向传播
batch_size, seq_len = 2, 5
x = torch.randint(0, 1000, (seq_len, batch_size))
state = model.begin_state(batch_size, x.device)
output, new_state = model(x, state)
print(f"\n输入: {x.shape}")
print(f"输出: {output.shape}")
print(f"隐状态: {new_state[0].shape}")
print(f"记忆元: {new_state[1].shape}")

---

## 第三部分: GRU (Gated Recurrent Unit)

### 3.1 GRU的动机

**LSTM的问题**:
- 参数多(4组权重)
- 计算慢
- 复杂度高

**GRU的改进**:
- 只有2个门(vs LSTM的3个)
- 无需记忆元
- 更快,参数更少
- 性能相当!

### 3.2 GRU的两个门

**1. 重置门** (Reset Gate): 忽略哪些历史?
$$\mathbf{R}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xr} + \mathbf{H}_{t-1} \mathbf{W}_{hr} + \mathbf{b}_r)$$

- $\mathbf{R}_t \approx 0$: 忽略历史,像新开始
- $\mathbf{R}_t \approx 1$: 保留全部历史

**2. 更新门** (Update Gate): 保留多少历史?
$$\mathbf{Z}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xz} + \mathbf{H}_{t-1} \mathbf{W}_{hz} + \mathbf{b}_z)$$

- $\mathbf{Z}_t \approx 0$: 完全更新为新值
- $\mathbf{Z}_t \approx 1$: 保持旧状态

### 3.3 GRU更新规则

**候选隐状态**:
$$\tilde{\mathbf{H}}_t = \tanh(\mathbf{X}_t \mathbf{W}_{xh} + (\mathbf{R}_t \odot \mathbf{H}_{t-1}) \mathbf{W}_{hh} + \mathbf{b}_h)$$

注意: $\mathbf{R}_t \odot \mathbf{H}_{t-1}$ 选择性使用历史

**最终隐状态**:
$$\mathbf{H}_t = \mathbf{Z}_t \odot \mathbf{H}_{t-1} + (1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t$$

- $\mathbf{Z}_t \odot \mathbf{H}_{t-1}$: 保留多少旧状态
- $(1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t$: 添加多少新状态

### 3.4 LSTM vs GRU对比

| 特性 | LSTM | GRU |
|------|------|-----|
| 门数量 | 3 (输入/遗忘/输出) | 2 (重置/更新) |
| 记忆机制 | 记忆元 $\mathbf{C}_t$ | 无,直接用 $\mathbf{H}_t$ |
| 参数量 | 4组权重 | 3组权重 |
| 计算速度 | 慢 | 快 |
| 性能 | 略优 | 相当 |
| 适用场景 | 长序列,需要精确记忆 | 一般序列,资源受限 |

### 3.5 实现GRU

In [ ]:
def get_gru_params(vocab_size, num_hiddens, device):
    """初始化GRU参数"""
    num_inputs = num_outputs = vocab_size
    
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    
    def three():
        return (normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))
    
    # 重置门
    W_xr, W_hr, b_r = three()
    # 更新门
    W_xz, W_hz, b_z = three()
    # 候选隐状态
    W_xh, W_hh, b_h = three()
    
    # 输出层
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    
    params = [W_xr, W_hr, b_r, W_xz, W_hz, b_z, W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params


def gru(inputs, state, params):
    """GRU前向传播"""
    W_xr, W_hr, b_r, W_xz, W_hz, b_z, W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    
    for X in inputs:
        # 重置门
        R = torch.sigmoid(torch.mm(X, W_xr) + torch.mm(H, W_hr) + b_r)
        # 更新门
        Z = torch.sigmoid(torch.mm(X, W_xz) + torch.mm(H, W_hz) + b_z)
        # 候选隐状态
        H_tilde = torch.tanh(torch.mm(X, W_xh) + torch.mm(R * H, W_hh) + b_h)
        # 更新隐状态
        H = Z * H + (1 - Z) * H_tilde
        # 输出
        Y = torch.mm(H, W_hq) + b_q
        outputs.append(Y)
    
    return torch.cat(outputs, dim=0), (H,)


# PyTorch的GRU
class GRUModel(nn.Module):
    """GRU语言模型"""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0.5):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, num_hiddens, num_layers,
                          dropout=dropout, batch_first=False)
        self.fc = nn.Linear(num_hiddens, vocab_size)
    
    def forward(self, x, state=None):
        x = self.embedding(x)
        output, state = self.gru(x, state)
        output = self.fc(output)
        return output, state
    
    def begin_state(self, batch_size, device):
        return torch.zeros((self.num_layers, batch_size, self.num_hiddens), device=device)


# 参数量对比
lstm_model = LSTMModel(1000, 128, 256, 2)
gru_model = GRUModel(1000, 128, 256, 2)

lstm_params = sum(p.numel() for p in lstm_model.parameters())
gru_params = sum(p.numel() for p in gru_model.parameters())

print(f"LSTM参数量: {lstm_params:,}")
print(f"GRU参数量: {gru_params:,}")
print(f"GRU减少: {(1 - gru_params / lstm_params) * 100:.1f}%")

---

## 第四部分: 实践建议

### 4.1 如何选择?

**使用RNN如果**:
- 序列很短(<20)
- 计算资源极其有限
- 只是快速原型

**使用GRU如果**:
- ✅ 一般长度序列(20-200)
- ✅ 需要快速训练
- ✅ GPU内存有限
- ✅ 默认选择!

**使用LSTM如果**:
- 序列很长(>200)
- 需要精确的长期记忆
- 充足的计算资源
- 追求最佳性能

### 4.2 超参数调优

**隐藏层大小**:
```python
# 推荐值
num_hiddens = [128, 256, 512, 1024]
# 规则: 不要超过词表大小
```

**层数**:
```python
# 1-3层通常够用
num_layers = [1, 2, 3]
# 更深不一定更好!
```

**Dropout**:
```python
# 防止过拟合
dropout = [0.2, 0.3, 0.5]
# 小数据用大dropout
```

**学习率**:
```python
# LSTM/GRU对学习率敏感
lr = [0.001, 0.002, 0.005]
# 使用学习率调度
```

### 4.3 常见陷阱

1. **忘记梯度裁剪**
   ```python
   # 必须!
   torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
   ```

2. **批次太小**
   ```python
   # 至少32
   batch_size >= 32
   ```

3. **序列太长**
   ```python
   # 截断长序列
   max_seq_len = 200
   ```

4. **状态未分离**
   ```python
   # 训练时
   state = state.detach()  # 截断梯度
   ```

---

## 小结

### RNN的演进

```
RNN (简单但有梯度问题)
  ↓
LSTM (3个门 + 记忆元,参数多但强大)
  ↓
GRU (2个门,简化但高效)
  ↓
Transformer (注意力机制,完全抛弃循环)
```

### 核心对比

| 模型 | 梯度问题 | 参数量 | 速度 | 性能 | 推荐度 |
|------|----------|--------|------|------|--------|
| RNN | ❌ 严重 | 少 | 快 | 差 | ⭐ |
| LSTM | ✅ 解决 | 多 | 慢 | 好 | ⭐⭐⭐⭐ |
| GRU | ✅ 解决 | 中 | 中 | 好 | ⭐⭐⭐⭐⭐ |

### 关键公式

**LSTM**:
$$\mathbf{C}_t = \mathbf{F}_t \odot \mathbf{C}_{t-1} + \mathbf{I}_t \odot \tilde{\mathbf{C}}_t$$
$$\mathbf{H}_t = \mathbf{O}_t \odot \tanh(\mathbf{C}_t)$$

**GRU**:
$$\mathbf{H}_t = \mathbf{Z}_t \odot \mathbf{H}_{t-1} + (1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t$$

### 实践建议

1. **默认用GRU**: 性价比最高
2. **长序列用LSTM**: 更强的记忆
3. **必须梯度裁剪**: 防止爆炸
4. **合理Dropout**: 防止过拟合
5. **监控梯度**: 及时发现问题

## 练习

1. 对比RNN/LSTM/GRU在相同数据上的性能
2. 可视化LSTM的门控值,观察学到的模式
3. 实现双向LSTM
4. 在长序列上测试梯度消失问题